In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

In [3]:
from load_data import load_dataset, summarize

df = load_dataset()

Loading: UNSW-NB15_1.csv
Loading: UNSW-NB15_2.csv
Loading: UNSW-NB15_3.csv
Loading: UNSW-NB15_4.csv


In [4]:
summarize(df)


Shape: 2540047 rows, 49 columns

Columns: 
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']

Class distribution ('Label'):
Label
0    2218764
1     321283
Name: count, dtype: int64

Class distribution ('attack_cat'):
attack_cat
NaN               2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Na

In [5]:
missing_by_col = df.isnull().sum()
missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
missing_by_col

attack_cat          2218764
is_ftp_login        1429879
ct_flw_http_mthd    1348145
dtype: int64

## Shortcut-learning check

Before we commit to a preprocessing strategy, we want evidence about two failure modes that Dr. Chuprov flagged in the proposal feedback:

- **Leakage** — already handled: `load_data.py` renames the mistyped `ct_src_ ltm` column, and the columns listed as *excluded* in `data/FEATURES.md` (srcip, dstip, Stime, Ltime, label, attack_cat) are deliberately **not** used as features.
- **Shortcut learning** — the question this section answers: *if we give a model only the 6 "suspicious" columns* (`sport`, `dsport`, `proto`, `service`, `state`, `is_sm_ips_ports`), *can it identify the attack type well enough that we'd be measuring dataset construction instead of attack behaviour?*

We test three feature groups (shortcut-only, behavioural-only, combined) under two training modes (standard, and with `class_weight="balanced"`), and score with **macro F1** and **rare-class F1** (the average F1 over every class except the majority one, which here is `Generic`). These are more honest metrics than accuracy, because the dataset is heavily imbalanced.

The function we call (`run_shortcut_check`) lives in `src/feature_selection.py`.


In [7]:
from feature_selection import run_shortcut_check

results = run_shortcut_check(df, target="attack_cat", max_rows=None)



=== SHORTCUT-LEARNING CHECK (target: attack_cat) ===
Rows used: 321,283 rows (all)

--- UNBALANCED (standard training) ---
  shortcut-only     
      accuracy  = 0.8731
      F1 macro  = 0.4943
      F1 rare   = 0.4331   <- excludes 'Generic'
  behavioural-only  
      accuracy  = 0.8953
      F1 macro  = 0.5566
      F1 rare   = 0.5022   <- excludes 'Generic'
  combined          
      accuracy  = 0.8994
      F1 macro  = 0.5982
      F1 rare   = 0.5489   <- excludes 'Generic'

--- BALANCED (class_weight='balanced') ---
  shortcut-only     
      accuracy  = 0.8317
      F1 macro  = 0.4819
      F1 rare   = 0.4187   <- excludes 'Generic'
  behavioural-only  
      accuracy  = 0.8592
      F1 macro  = 0.5763
      F1 rare   = 0.5247   <- excludes 'Generic'
  combined          
      accuracy  = 0.8600
      F1 macro  = 0.6031
      F1 rare   = 0.5548   <- excludes 'Generic'

--- Per-class F1 for behavioural-only (unbalanced -> balanced) ---
      Backdoor             0.1174 -> 0.0320


### How to read the numbers

Look first at the **rare-class F1** column (the one that excludes `Generic`).

- **Shortcut-only F1 (rare)** — this is the "shortcut signal". If it were close to 1.0, the model could identify the attack type almost entirely from the 6 flagged columns, which would be a serious shortcut-learning red flag. A low-to-moderate value (as we expect) means the shortcut columns contribute *something*, but the model still needs the behavioural features.
- **Behavioural-only F1 (rare)** — the "honest" model. This is the number we ultimately care about for the main model.
- **Balanced vs. unbalanced** — comparing the two sections tells us whether simple class-weighting helps or hurts the rare attack classes. On UNSW-NB15, we expect a **mixed result**: some rare classes (Worms) improve, others (Backdoor, Analysis) can get worse. That's a genuinely useful finding — it tells us a blanket `class_weight="balanced"` is not the right answer, and motivates the more targeted strategies we'll try in notebook 02 (per-class oversampling, focal loss, threshold tuning).

**Bottom line for the proposal:** shortcut learning is *modest*, not dominant. The main challenge is the rare attack classes, and simple balancing is a mixed bag — which is exactly the kind of evidence that justifies moving to a preprocessing notebook (02) with a more deliberate class-imbalance strategy.
